[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: seguridad en GitHub Actions

*Nota para quien imparte el curso: el cuaderno 13 contiene una sola actividad explícita (buscar secretos con *gitleaks*; el enunciado termina en dos puntos y las órdenes están en la celda siguiente). Los ejercicios 2 a 5 verifican, con código, afirmaciones que el cuaderno hace sobre los *workflows* del curso y que no pide comprobar.*

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Buscar secretos en el árbol de trabajo y en el historial

Buscar secretos en el árbol de trabajo y en el historial del repositorio con *gitleaks*:

```bash
gitleaks dir --redact .
gitleaks git --redact --log-opts=--all .
```

*gitleaks* no viene instalado en el contenedor de desarrollo del curso (`.devcontainer/Dockerfile`). Se instala con el paquete oficial de la herramienta o con el binario de su página de versiones. Como no siempre está disponible, la solución ofrece dos caminos: la herramienta real (celda con etiqueta) y un detector sencillo en *Python* que sí se ejecuta en cualquier equipo.

La siguiente celda definirá `buscar_secretos(raiz)`, un detector de patrones conocidos: cabeceras de claves privadas, *tokens* de *GitHub*, claves de acceso de *AWS* y claves de *API* de *Google*, y el campo `private_key` de las llaves *JSON* de cuentas de servicio. Recorre los archivos de texto, ignora `.git` y devuelve una lista de `(archivo, línea, patrón)` sin mostrar nunca el valor encontrado.

In [ ]:
import re
from pathlib import Path

PATRONES = {
    "clave privada": re.compile(r"-----BEGIN (?:[A-Z]+ )?PRIVATE KEY-----"),
    "token de GitHub": re.compile(r"\b(?:ghp|gho|ghu|ghs|ghr)_[A-Za-z0-9]{36}\b|\bgithub_pat_[A-Za-z0-9_]{50,}\b"),
    "clave de acceso de AWS": re.compile(r"\bAKIA[0-9A-Z]{16}\b"),
    "clave de API de Google": re.compile(r"\bAIza[0-9A-Za-z_-]{35}\b"),
    "llave JSON de cuenta de servicio": re.compile(r'"private_key"\s*:\s*"-----BEGIN'),
}
OMITIR = {".git", ".venv", "venv-api", "__pycache__", "actions-runner", "node_modules"}

def buscar_secretos(raiz):
    hallazgos = []
    for ruta in sorted(Path(raiz).rglob("*")):
        if not ruta.is_file() or OMITIR & set(ruta.parts):
            continue
        try:
            texto = ruta.read_text(encoding="utf-8")
        except (UnicodeDecodeError, OSError):
            continue  # archivo binario o ilegible
        for numero, linea in enumerate(texto.splitlines(), start=1):
            for nombre, patron in PATRONES.items():
                if patron.search(linea):
                    hallazgos.append((str(ruta.relative_to(raiz)), numero, nombre))
    return hallazgos

La siguiente celda comprobará que el detector funciona. Crea un archivo temporal con valores **falsos** construidos al vuelo (no hay ningún valor sensible en el cuaderno) y verifica que los detecta; después comprueba que un archivo sin secretos no produce hallazgos.

In [ ]:
prueba = u.directorio_temporal()
(prueba / "falso.txt").write_text(
    "token = " + "ghp_" + "a" * 36 + "\n" + "-----BEGIN " + "PRIVATE KEY-----\n", encoding="utf-8")
(prueba / "limpio.txt").write_text("nada que ocultar\n", encoding="utf-8")

hallazgos = buscar_secretos(prueba)
print(hallazgos)
assert {h[2] for h in hallazgos} == {"token de GitHub", "clave privada"}
assert all(h[0] == "falso.txt" for h in hallazgos)

La siguiente celda aplicará el detector al árbol de trabajo del repositorio. Se espera una lista vacía.

In [ ]:
hallazgos = buscar_secretos(RAIZ)
print("Hallazgos:", hallazgos)
assert hallazgos == []

La siguiente celda ejecutará *gitleaks* sobre el árbol de trabajo y sobre todo el historial. `--redact` oculta el valor de cualquier hallazgo en la salida. Requiere tener instalado `gitleaks`. **Nota:** esta celda lleva la etiqueta `skip-execution` y la CI la omite a propósito. `gitleaks git` analiza el historial de *Git* y la CI trabaja sobre una copia del repositorio sin la carpeta `.git`, de modo que no encontraría ningún *commit* que revisar; además, `gitleaks` no está instalado en el *devcontainer*. Para ejecutarla hacen falta un clon con su historial y `gitleaks` instalado.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
gitleaks dir --redact .
gitleaks git --redact --log-opts=--all .

**Salida esperada.** Cada comando termina con el código 0 y la línea `no leaks found`; el segundo indica además cuántos *commits* se analizaron. Si hubiera un hallazgo, el código de salida es 1 y la salida indica la regla, el archivo y la línea, pero el secreto aparece oculto. Un secreto que alguna vez se subió debe considerarse comprometido y **revocarse**, aunque se borre del archivo.

## Ejercicio 2. Comprobar que las acciones están fijadas por SHA

El cuaderno afirma que cada *action* de los *workflows* del curso se fija por el *SHA* completo de su *commit*, con un comentario que indica la versión (por ejemplo, `# v7.0.1`).

La siguiente celda recorrerá todos los *workflows* de `.github/workflows`, listará cada acción usada y comprobará que su referencia es un *SHA* de 40 caracteres hexadecimales y que la línea lleva un comentario con la versión.

In [ ]:
import re

DIRECTORIO = RAIZ / ".github" / "workflows"
total = 0
for archivo in sorted(DIRECTORIO.glob("*.yml")):
    wf = u.cargar_workflow(archivo)
    sin_fijar = u.acciones_sin_fijar(wf)
    assert sin_fijar == [], (archivo.name, sin_fijar)
    for linea in archivo.read_text(encoding="utf-8").splitlines():
        if re.match(r"\s*(- )?uses:\s*[^./]", linea):
            total += 1
            assert re.search(r"@[0-9a-f]{40}\s+# v\d", linea), linea
    print(f"{archivo.name:25} {len(u.acciones_usadas(wf))} acciones, todas fijadas")
print("Total de líneas `uses` revisadas:", total)

## Ejercicio 3. Comprobar los permisos mínimos

El cuaderno afirma que los *workflows* declaran `permissions:` explícitos y mínimos: por defecto `contents: read`; solo el *job* que publica la imagen pide `packages: write`, y solo el de *Cloud Run* pide `id-token: write`.

La siguiente celda leerá los permisos de cada *workflow*: los globales y los de cada *job*. Comprobará que existen permisos globales con `contents: read` y que los permisos de escritura aparecen únicamente donde el cuaderno dice. Si se añade otro *workflow* que necesite `id-token: write` (por ejemplo, para una prueba en vivo), hay que actualizar el conjunto `PERMITIDOS`.

In [ ]:
PERMITIDOS = {
    ("ci.yml", "publicar-imagen", "packages"),
    ("deploy-cloud-run.yml", "desplegar", "id-token"),
}
escrituras = set()
for archivo in sorted(DIRECTORIO.glob("*.yml")):
    wf = u.cargar_workflow(archivo)
    globales = wf.get("permissions")
    assert globales is not None, f"{archivo.name}: sin permissions globales"
    assert globales.get("contents") == "read", archivo.name
    assert not any(valor == "write" for valor in globales.values()), archivo.name
    for job, definicion in wf["jobs"].items():
        for permiso, valor in definicion.get("permissions", {}).items():
            if valor == "write":
                escrituras.add((archivo.name, job, permiso))
print("Permisos de escritura:", sorted(escrituras))
assert escrituras <= PERMITIDOS, escrituras - PERMITIDOS

## Ejercicio 4. Comprobar que el checkout no conserva el token

El cuaderno afirma que el *checkout* usa `persist-credentials: false` para que el *token* no quede en `.git/config`.

La siguiente celda buscará todos los pasos que usan `actions/checkout` y comprobará que cada uno declara `persist-credentials: false`.

In [ ]:
revisados = 0
for archivo in sorted(DIRECTORIO.glob("*.yml")):
    wf = u.cargar_workflow(archivo)
    for job, definicion in wf["jobs"].items():
        for paso in definicion["steps"]:
            if paso.get("uses", "").startswith("actions/checkout@"):
                revisados += 1
                assert paso.get("with", {}).get("persist-credentials") is False, (archivo.name, job)
print("Pasos de checkout revisados:", revisados)
assert revisados >= 4

## Ejercicio 5. Comprobar las defensas contra la inyección de comandos y la actualización automática

El cuaderno afirma que el despliegue por *SSH* envía el guion remoto entre comillas (`<<'REMOTO'`) y que `.github/dependabot.yml` abre *pull requests* para las *actions*, las dependencias de *Python* y la imagen base.

La siguiente celda comprobará la comilla del *heredoc* en `deploy-linux-ssh.yml` y los tres ecosistemas de `dependabot.yml`.

In [ ]:
import yaml

ssh = (DIRECTORIO / "deploy-linux-ssh.yml").read_text(encoding="utf-8")
assert "<<'REMOTO'" in ssh and "<<REMOTO" not in ssh
print("El guion remoto va entre comillas")

dependabot = yaml.safe_load((RAIZ / ".github" / "dependabot.yml").read_text(encoding="utf-8"))
ecosistemas = {entrada["package-ecosystem"] for entrada in dependabot["updates"]}
print("Ecosistemas:", sorted(ecosistemas))
assert {"github-actions", "pip", "docker"} <= ecosistemas

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>